In [10]:
import pandas as pd
import numpy as np
import os
from collections import Counter
import warnings
from Bio import SeqIO
from Bio.Blast import NCBIWWW
from Bio.Blast import NCBIXML
warnings.filterwarnings("ignore")
import time
import random

In [5]:
#AEJK#TE7679
query_id='AEJK#TE7679'
query='AGTGCACCCCTCTTTGCTATATAAATAAGACCTCTTTTATTCATTAAAAAAAAAAAGGACACAACTCATACAATCATTCACTCTCAATCACAAGTTCTCTCTTCATTCTCTCATTGCTCTCTTGCTACTCTAAATACACTAAGATTTCGATAAACATTCAAGTGTTCTTCTTCGCTACTTTGCCATTTTGCTTTTGTTCGAGTAAAGGTTGGATCAACTTGCTTGCATTGCTACCATTCCTAATCTACTTAACCGTTAGTTGATTGCTTTGACAGGTTTCAAACTTGAAGTTCAAGTTGAAGATTAATTGAAGAAAGATTTATTTATTTATTTGAATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATTCTTATTTTATTTATTCGAATGTTGTAACATTTGTAACTCTAAAGATCATATATATAAATTTATTTGAGGGGTCGTCTAGGGGAGGGGGATTTACATGCCTACTTGTTCATTGTAAAATTTTTAGAAATCATGCCTATAGGTTTGTCTTTAACCACCTAGAATTATTGTTTGTAGGTATTATAATCCAATCAATTGTTTGATGCTTTGCTAATAAGTCTAAAAATAATAAACTAGATTCCATTATTTTAACGTTAAAATCAAATCATAATAGGCTGATTAGGTGTTTTAACAATGTTATAACTTTTAGCATCATCTTGTCATGTAGAAACCATGCCTAAGGATTAAACTGTTCATATCATTTAGATTTACAAAATCATGCATATATGTGTTTCTCA'

In [7]:
def query_NCBI_NR(query_id,query,save_file):
    result_handle = NCBIWWW.qblast("blastn", "nt", query, entrez_query="Viruses[Organism]")

    # 将BLAST结果保存到本地文件
    with open(save_file, "w") as out_handle:
        out_handle.write(result_handle.read())
        # 关闭结果句柄
        result_handle.close()
        print(query_id," BLAST搜索完成")
save_file=query_id+'_blast_results.xml'
query_NCBI_NR(query_id,query,save_file)

AEJK#TE7679  BLAST搜索完成


In [17]:
def parse_NCBI_BLAST_Result(query_id,save_file):
    with open(save_file, "r") as result_file:
        blast_records = NCBIXML.parse(result_file)
        Query_Title=[]
        Virus_Title=[]
        Virus_Length=[]
        Score=[]
        Align_length=[]
        Bits=[]
        Expect=[]
        Gaps=[]
        Identities=[]
        Query=[]
        Q_start=[]
        Q_end=[]
        S_start=[]
        S_end=[]
        # 遍历每个BLAST记录
        for blast_record in blast_records:
            # 遍历每个比对结果
            for alignment in blast_record.alignments:
                # 遍历每个高得分片段对（HSP）
                for hsp in alignment.hsps:
                    title=alignment.title
                    length=alignment.length
                    score=hsp.score
                    align_length=hsp.align_length
                    bits=hsp.bits
                    expect=hsp.expect
                    gaps=hsp.gaps
                    identities=hsp.identities
                    q_start=hsp.query_start
                    q_end=hsp.query_end
                    query=hsp.query
                    s_start=hsp.sbjct_start
                    s_end=hsp.sbjct_end
                    Virus_Title.append(title)
                    Query_Title.append(query_id)
                    Virus_Length.append(length)
                    Score.append(score)
                    Align_length.append(align_length)
                    Bits.append(bits)
                    Expect.append(expect)
                    Gaps.append(gaps)
                    Identities.append(identities)
                    Query.append(query)
                    Q_start.append(q_start)
                    Q_end.append(q_end)
                    S_start.append(s_start)
                    S_end.append(s_end)
    data=pd.DataFrame(Query_Title,columns=['Query_Title'])
    data['Virus_Title']=pd.DataFrame(Virus_Title)
    data['Virus_Length']=pd.DataFrame(Virus_Length)
    data['S_start']=pd.DataFrame(S_start)
    data['S_end']=pd.DataFrame(S_end)
    data['Score']=pd.DataFrame(Score)
    data['Align_length']=pd.DataFrame(Align_length)
    data['Bits']=pd.DataFrame(Bits)
    data['Expect']=pd.DataFrame(Expect)
    data['Gaps']=pd.DataFrame(Gaps)
    data['Identities']=pd.DataFrame(Identities)
    data['Q_start']=pd.DataFrame(Q_start)
    data['Q_end']=pd.DataFrame(Q_end)
    data['Query']=pd.DataFrame(Query)
    return data

In [19]:
data=parse_NCBI_BLAST_Result(query_id,save_file)
data[data['Virus_Title'].apply(lambda x:1 if 'tomato' in x.lower() else 0)==1].head()

,Query_Title,Virus_Title,Virus_Length,S_start,S_end,Score,Align_length,Bits,Expect,Gaps,Identities,Q_start,Q_end,Query
25,AEJK#TE7679,gi|2379690574|gb|OM469321.1| Tomato zonate spo...,3253,2245,2325,110.0,81,100.4720,1.074720e-16,3,71,318,395,ATTTATTTATTTATTTGAATATATATATATATATATATATATATAT...
26,AEJK#TE7679,gi|2379690574|gb|OM469321.1| Tomato zonate spo...,3253,2231,2327,103.0,97,94.1598,1.595030e-14,4,81,318,410,ATTTATTTATT--TATTTGAATATATATATATATATATATATATAT...
28,AEJK#TE7679,gi|2638408832|gb|OR911350.1| Tomato zonate spo...,3336,2340,2415,103.0,76,94.1598,1.595030e-14,1,67,321,395,TATTTATTTATTTGAATATATATATATATATATATATATATATATA...
